In [4]:
import psycopg
import os
from dotenv import load_dotenv

In [11]:
load_dotenv()
dbname = os.getenv("DB_DATABASE")


print('Database health report')
print('-'*30, '\n')
try:
    with psycopg.connect(
        host=os.getenv("DB_HOST"),
        dbname=dbname,
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT"),
        connect_timeout=5
    ) as connection:

        with connection.cursor() as cursor:

#number of connections on database
            
            cursor.execute("""
                SELECT count(*)
                FROM pg_stat_activity;
            """)

            connection_count = cursor.fetchone()[0]
            print(f"Number of connections on database {dbname} is {connection_count}.")

#maximal connections allowed
            
            cursor.execute("""
                SELECT setting 
                FROM pg_settings 
                WHERE name = 'max_connections';
            """)

            max_connections= cursor.fetchone()[0]
            print(f"Maximal connections allowed: {max_connections}.")
            print('-'*50)

#database size
            
            cursor.execute("""
                SELECT pg_size_pretty(pg_database_size(%s));
            """, (dbname,))

            db_size = cursor.fetchone()[0]
            print('Size of database is:', db_size)
            print('-'*50)

#tables ordered by size
            
            cursor.execute("""
                SELECT relname,
                pg_size_pretty(pg_total_relation_size(relid))
                FROM pg_catalog.pg_statio_user_tables
                ORDER BY pg_total_relation_size(relid) DESC;
            """)
            
            print(f"Tables ordered by size in database {dbname}:\n")
            for row in cursor.fetchall():
                print (row[0],row[1])
            print('-'*50)

#when was server's start_time
            
            cursor.execute("""
                SELECT pg_postmaster_start_time()
            """)
            server_start_time = cursor.fetchone()[0]
            print('Server last started on:', server_start_time.strftime("%d.%m.%Y %H:%M:%S"))
            print('-'*50)

#how long is server up
            cursor.execute("""
                SELECT extract(day from now() - pg_postmaster_start_time()) || ' days ' ||
                to_char(now() - pg_postmaster_start_time(), 'HH24:MI:SS');
            """)

            server_uptime=cursor.fetchone()[0]
            print('Server is up for:', server_uptime)
                   


except psycopg.OperationalError as e:
    print('Connection error:', e)
except psycopg.Error as e:
    print('Database error:', e)


Database health report
------------------------------ 

Number of connections on database pg4e_3fb3abd7f7 is 25.
Maximal connections allowed: 200.
--------------------------------------------------
Size of database is: 8797 kB
--------------------------------------------------
Tables ordered by size in database pg4e_3fb3abd7f7:

tickets 96 kB
track_raw 64 kB
car_make 48 kB
car_model 48 kB
pg4e_meta 48 kB
course 40 kB
make 40 kB
roster 40 kB
student 40 kB
pg4e_debug 32 kB
pg4e_result 32 kB
automagic 24 kB
model 24 kB
car 24 kB
ages 8192 bytes
--------------------------------------------------
Server last started on: 23.05.2025 14:47:56
--------------------------------------------------
Server is up for: 290 days 21:18:34


In [10]:
# Script using a single SQL query

load_dotenv()
dbname = os.getenv("DB_DATABASE")

print('Database health report')
print('-'*40,'\n')
try:
    with psycopg.connect(
        host=os.getenv("DB_HOST"),
        dbname=dbname,
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT"),
        connect_timeout=5
    ) as connection:

        with connection.cursor() as cursor:

#number of connections on database
            
            cursor.execute("""
                SELECT
            current_database(),
            pg_size_pretty(pg_database_size(current_database())),
            (SELECT count(*) FROM pg_stat_activity),
            (SELECT setting FROM pg_settings WHERE name = 'max_connections'),
            pg_postmaster_start_time(),
            (SELECT extract(day from now() - pg_postmaster_start_time()) || ' days ' ||
            to_char(now() - pg_postmaster_start_time(), 'HH24:MI:SS')
            );
            """)
            
            database, db_size, connection_count, max_connections, server_started_time, server_uptime = cursor.fetchone()
            print('Database name:', database)
            print('Database size:', db_size)
            print('Number of connections:', connection_count)
            print('Max connections:', max_connections)
            print('Server started on:', server_started_time.strftime("%d.%m.%Y %H:%M:%S"))
            print('Server uptime:', server_uptime)

except psycopg.OperationalError as e:
    print('Connection error:', e)
except psycopg.Error as e:
    print('Database error:', e)  


Database health report
---------------------------------------- 

Database name: pg4e_3fb3abd7f7
Database size: 8797 kB
Number of connections: 25
Max connections: 200
Server started on: 23.05.2025 14:47:56
Server uptime: 290 days 21:18:00
